# Orientation Bug Replication

Creates synthetic ellipse polygons at **known** angles and runs them through both orientation paths:

1. **`fitEllipse`** (scikit-image `EllipseModel`) → returns `theta` in radians
2. **`boulder_row`** (Shapely minimum rotated rectangle → `azimuth()`) → returns `angle`/`angle180`

For each input angle we compare what the code returns vs. what it should return.
If a column is constant regardless of input → that path contains the bug.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely import segmentize

from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row, azimuth

## Helper: generate a synthetic ellipse polygon at a known angle

In [ ]:
def make_ellipse_polygon(a, b, theta_degrees, cx=0.0, cy=0.0, n_pts=200):
    """
    Return a Shapely Polygon of an ellipse with:
      a            : semi-major axis length
      b            : semi-minor axis length  (b < a)
      theta_degrees: rotation CCW from positive x-axis (East)
      cx, cy       : center coordinates
    """
    theta_rad = np.radians(theta_degrees)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    x = a * np.cos(t)
    y = b * np.sin(t)
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad) + cx
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad) + cy
    return Polygon(np.column_stack([x_rot, y_rot]))


def make_noisy_ellipse_polygon(a, b, theta_degrees, cx=0.0, cy=0.0, n_pts=40, noise_scale=0.05):
    """
    Like make_ellipse_polygon but with random noise added to boundary points.
    Simulates a real irregular boulder outline rather than a clean parametric shape.
    noise_scale is fraction of semi-minor axis b.
    """
    rng = np.random.default_rng(seed=42)
    theta_rad = np.radians(theta_degrees)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    noise = rng.normal(0, noise_scale * b, size=n_pts)
    r = 1.0 + noise
    x = a * np.cos(t) * r
    y = b * np.sin(t) * r
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad) + cx
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad) + cy
    return Polygon(np.column_stack([x_rot, y_rot]))

## Test parameters

In [ ]:
A   = 15.0      # semi-major axis
B   = 10.0      # semi-minor axis  →  a/b = 1.5
CX  = 500.0
CY  = 500.0
RES = 1.0       # resolution used in segmentize step (same role as in shp_geom.ellipse)

test_angles = [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165]

print(f"Ellipse: a={A}, b={B}, a/b={A/B:.2f}, center=({CX}, {CY})")

---
## Part 1: Raw paths (independent)
Tests `fitEllipse` and `boulder_row` on the clean synthetic ellipse directly,
**without** chaining them together.

In [ ]:
results = []

for deg in test_angles:
    poly = make_ellipse_polygon(A, B, deg, cx=CX, cy=CY)
    row  = pd.Series({"geometry": poly})

    # Path 1: fitEllipse
    try:
        _, a_fit, b_fit, theta_rad = fitEllipse(row)
        theta_deg = np.degrees(theta_rad)
    except Exception as e:
        theta_deg, a_fit, b_fit = None, None, None
        print(f"fitEllipse failed at {deg}°: {e}")

    # Path 2: boulder_row on MRR of the original polygon
    try:
        mrr_poly = poly.minimum_rotated_rectangle
        mrr_row  = pd.Series({"geometry": mrr_poly})
        (_, _, _, _, _, angle_default, angle360, angle180) = boulder_row(mrr_row)
    except Exception as e:
        angle_default, angle360, angle180 = None, None, None
        print(f"boulder_row failed at {deg}°: {e}")

    results.append({
        "input_deg"           : deg,
        "fitEllipse_theta_deg": theta_deg,
        "fitEllipse_a"        : a_fit,
        "fitEllipse_b"        : b_fit,
        "mrr_angle_default"   : angle_default,
        "mrr_angle180"        : angle180,
    })

df = pd.DataFrame(results)
df

In [ ]:
expected = [d % 180 for d in test_angles]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[0].plot(test_angles, df["fitEllipse_theta_deg"] % 180, 'ro-', label='fitEllipse theta')
axes[0].set_title("Path 1: fitEllipse (EllipseModel)")
axes[0].set_xlabel("Input angle (°)"); axes[0].set_ylabel("Returned angle (°)")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[1].plot(test_angles, df["mrr_angle180"], 'bs-', label='MRR angle180')
axes[1].set_title("Path 2: boulder_row (MRR + azimuth)")
axes[1].set_xlabel("Input angle (°)"); axes[1].set_ylabel("Returned angle (°)")
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig("orientation_part1_raw_paths.png", dpi=150)
plt.show()

---
## Part 2: Full post-processing pipeline (chained)

Replicates exactly what the real pipeline does in `shp_geomorph.boulder(is_boulder=False)`:

```
raw boulder polygon
  → segmentize (adds boundary points at resolution spacing)
  → fitEllipse → fitted ellipse polygon
  → minimum_rotated_rectangle
  → boulder_row → final reported angle
```

Tested on both a **clean** synthetic ellipse and a **noisy** one (closer to real boulder outlines).

In [ ]:
def full_pipeline(poly, res=RES):
    """
    Run the full orientation pipeline on a single polygon.
    Returns dict with theta from fitEllipse and angle180 from boulder_row.
    """
    # Step 1: segmentize (same as shp_geom.ellipse does before fitting)
    poly_seg = segmentize(poly, res)
    row_seg  = pd.Series({"geometry": poly_seg})

    # Step 2: fit ellipse
    ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
    theta_deg = np.degrees(theta_rad)

    # Step 3: MRR of fitted ellipse polygon
    mrr_poly = ellipse_poly.minimum_rotated_rectangle
    mrr_row  = pd.Series({"geometry": mrr_poly})

    # Step 4: boulder_row → final orientation
    (_, _, long_axis, short_axis, _, angle_default, angle360, angle180) = boulder_row(mrr_row)
    aspect_ratio = long_axis / short_axis if short_axis > 0 else None

    return {
        "fitEllipse_theta_deg": theta_deg,
        "fitEllipse_a"        : a_fit,
        "fitEllipse_b"        : b_fit,
        "final_angle_default" : angle_default,
        "final_angle180"      : angle180,
        "aspect_ratio"        : aspect_ratio,
    }

In [ ]:
results_clean = []
results_noisy = []

for deg in test_angles:
    # Clean synthetic ellipse
    poly_clean = make_ellipse_polygon(A, B, deg, cx=CX, cy=CY)
    try:
        r = full_pipeline(poly_clean)
    except Exception as e:
        print(f"Clean pipeline failed at {deg}°: {e}")
        r = {k: None for k in ["fitEllipse_theta_deg","fitEllipse_a","fitEllipse_b","final_angle_default","final_angle180","aspect_ratio"]}
    results_clean.append({"input_deg": deg, **r})

    # Noisy polygon (simulates real irregular boulder outline)
    poly_noisy = make_noisy_ellipse_polygon(A, B, deg, cx=CX, cy=CY)
    try:
        r = full_pipeline(poly_noisy)
    except Exception as e:
        print(f"Noisy pipeline failed at {deg}°: {e}")
        r = {k: None for k in ["fitEllipse_theta_deg","fitEllipse_a","fitEllipse_b","final_angle_default","final_angle180","aspect_ratio"]}
    results_noisy.append({"input_deg": deg, **r})

df_clean = pd.DataFrame(results_clean)
df_noisy = pd.DataFrame(results_noisy)

print("=== Clean ellipse input ===")
print(df_clean.to_string())
print()
print("=== Noisy polygon input ===")
print(df_noisy.to_string())

In [ ]:
expected = [d % 180 for d in test_angles]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Clean — fitEllipse theta
axes[0, 0].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[0, 0].plot(test_angles, df_clean["fitEllipse_theta_deg"] % 180, 'ro-', label='fitEllipse theta')
axes[0, 0].set_title("Clean input: fitEllipse theta")
axes[0, 0].legend(); axes[0, 0].grid(True)

# Clean — final angle180
axes[0, 1].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[0, 1].plot(test_angles, df_clean["final_angle180"], 'bs-', label='final angle180')
axes[0, 1].set_title("Clean input: final angle180 (full pipeline)")
axes[0, 1].legend(); axes[0, 1].grid(True)

# Noisy — fitEllipse theta
axes[1, 0].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[1, 0].plot(test_angles, df_noisy["fitEllipse_theta_deg"] % 180, 'ro-', label='fitEllipse theta')
axes[1, 0].set_title("Noisy input: fitEllipse theta")
axes[1, 0].set_xlabel("Input angle (°)"); axes[1, 0].set_ylabel("Returned angle (°)")
axes[1, 0].legend(); axes[1, 0].grid(True)

# Noisy — final angle180
axes[1, 1].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[1, 1].plot(test_angles, df_noisy["final_angle180"], 'bs-', label='final angle180')
axes[1, 1].set_title("Noisy input: final angle180 (full pipeline)")
axes[1, 1].set_xlabel("Input angle (°)"); axes[1, 1].set_ylabel("Returned angle (°)")
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig("orientation_part2_full_pipeline.png", dpi=150)
plt.show()
print("Plot saved to orientation_part2_full_pipeline.png")

---
## Part 2 interpretation

**If clean and noisy inputs both vary correctly:** the bug is not in these functions — look further downstream (filtering, angle remapping, visualization).

**If noisy input goes flat but clean does not:** `fitEllipse` is unstable on irregular polygons — real boulder outlines are causing inconsistent fits.

**If `final_angle180` is flat but `fitEllipse_theta_deg` is not:** the bug is in `boulder_row`/`azimuth` when applied to the fitted ellipse polygon.

**If both are flat:** there is a systematic issue in `fitEllipse` itself regardless of input shape.